In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import time
from IPython.display import display, clear_output

In [ ]:
%%capture
%run "/content/drive/MyDrive/Colab Notebooks/Technology News Insight Engine/Projects/Technology News Insight Engine-Tools.ipynb"

In [ ]:
%%capture
!pip install bertopic sentence_transformers

In [ ]:
path = "/content/drive/MyDrive/Colab Notebooks/Technology News Insight Engine/Data/Hackernoon_uncleaned_all_subsets.csv"
df = pd.read_csv(path)

In [ ]:
path = "/content/drive/MyDrive/Colab Notebooks/Technology News Insight Engine/Data/results2.csv"
preprocessed_df = pd.read_csv(path)

In [ ]:
preprocessed_df.article = preprocessed_df.article.astype(str)

In [ ]:
class TechNewsInsightEngine:
  def __init__(self, article_df,
               api_key, groq_model,
               n_neighbors = 15, n_components = 5,
               min_cluster_size = 150, top_n_keywords = 30,
               embedding_model = ""):

    # Variables
    # original article df
    self.article_df = article_df

    # cleaned article df
    self.preprocessed_article_df = None

    # doucment info from bertopic
    self.berTopic_df = None
    self.bertTopic_articles = None

    # Preprocessing
      # clean text
    self.text_preocessor = textCleaner()

    # Summarization
      # frequency word summarization
    self.freqSum = frequencySummarizer()
      # text rank summarization
    self.textRankSum = textRankSummarizer()

    # BERTopic submodels and model
    self.berTopic_model = None
    self.n_neighbors = n_neighbors
    self.n_components = n_components
    self.min_cluster_size = min_cluster_size
    self.top_n_keywords = top_n_keywords
    self.embedding_model = embedding_model

     # Dimensionality Reudction
    self.umap_model = UMAP(
        n_neighbors = self.n_neighbors,
        n_components = self.n_components,
        min_dist = 0.0,
        metric = "cosine",
        random_state=42
        )

      # Clustering
    self.hdbscan_model = HDBSCAN(
        min_cluster_size = self.min_cluster_size,
        metric = "euclidean",
        cluster_selection_method = "eom",
       prediction_data = True
        )

    self.top_n_keywords = top_n_keywords

    self.representation_model = {
        "KeyBERT": KeyBERTInspired(top_n_words = self.top_n_keywords),

        "MMR": MaximalMarginalRelevance(top_n_words = self.top_n_keywords, diversity = 0.3),

        "POS": PartOfSpeech("en_core_web_sm",
                            top_n_words = self.top_n_keywords,
                            pos_patterns = [
                                [{'POS': 'ADJ'}, {'POS': 'NOUN'}],
                                 [{'POS': 'NOUN'}], [{'POS': 'ADJ'}]
                                ])
        }

    self.embedding_model = SentenceTransformer(embedding_model)
    self.embeddings = None

    # Groq
    self.api_key = api_key
    self.groq_model = groq_model




  # Functions / Methods
  # Preprocessing
  def preprocess_article_fit():
    self.articles["article"] = self.article_df["article"].astype(str)

    preprocessed_articles = []
    index = 0
    for article in self.articles:
      clear_output(wait=True)
      preprocessed_article = self.preprocessing(article)
      preprocessed_articles.append(preprocessed_article)
      display(index)
      index += 1

    self.preprocessed_article_df = self.article_df[["id", "companyName", "published_at", "title"]].copy()
    self.preprocessed_article_df["article"] = preprocessed_articles
    self.removeEmptyArticles(value = " ")



  def removeEmptyArticles(self, value):
    indices = np.where(self.preprocessed_article_df.article == value)[0]
    self.article_df = self.article_df.drop(indices).reset_index(drop=True)
    self.preprocessed_article_df = self.preprocessed_article_df.drop(indices).reset_index(drop=True)



  def removeOriginalArticles(self):
    id = self.preprocessed_article_df.id
    indices = np.where(self.article_df.id.isin(id))[0]
    self.article_df = self.article_df.loc[indices].reset_index(drop=True)



  def joinTokens(self, preprcoessed_token):
    sentences = [" ".join(sentence) for sentence in preprcoessed_token]
    preprocessed_article = " ".join(sentences)
    return preprocessed_article



  def get_num_tokens(self, preprocessed_article):
    preprocessed_article_splited = preprocessed_article.split()
    num_token = len(preprocessed_article_splited)
    return num_token



  # Summarize article by Frequency Word Summarization and Text Rank Summarization
  def summarization(self, article):
      freqSummarized_article = self.freqSum.summarize(article)
      summarized_token = self.text_preocessor.clean(freqSummarized_article)
      summarized_article = self.joinTokens(summarized_token)
      num_token = self.get_num_tokens(summarized_article)

      if num_token > 1:
        textRanked_article = self.textRankSum.textSummarize(summarized_token)
        return textRanked_article
      else:
        return summarized_article



  # Preprcoessing Tech News Articles
  def preprocessing(self, article):
    preprcoessed_token = self.text_preocessor.clean(article)
    preprocessed_article = self.joinTokens(preprcoessed_token)

    num_token = self.get_num_tokens(preprocessed_article)

    if num_token > 512:
      summarized_article = self.summarization(article)
      return summarized_article

    return preprocessed_article




  # Concatenate keywords from KeyBERT, MMR, and partOfSpeerch
  def concatKeywords(self, info):
    keyBert_keywords = info.KeyBERT.to_list()
    mmr_keywords = info.MMR.to_list()
    pos_keywords = info.POS.to_list()
    averaged_keywords = []

    for i in range(len(info)):
      keywords = keyBert_keywords[i] + mmr_keywords[i] + pos_keywords[i]
      keywords = list(set(keywords))
      keywords = " ".join(keywords)
      averaged_keywords.append(keywords)

    return averaged_keywords



  # Topic modeling fit
  def topic_fit(self):
    index = 1
    results = []
    cluster_size = 150
    repeat = True

    for _ in range(1):
      while repeat:
        clear_output(wait = True)

        self.bertTopic_fit()
        print("Groq begins to create topic names")
        results = self.groqTopic_fit()
        print("Groq ends")

        print("Remove Articles with NA Topic Name")
        self.dropNATopics()

        print("Assign Topic name to preprocessed article data frame")
        self.preprocessed_article_df["Topic"] = self.berTopic_df.CustomName

        print("Remove article from original data frame")
        self.removeOriginalArticles()

        clear_output(wait = True)
        print("Topic Modeling {}".format(index))

        index +=1

        if "NA" not in list(results):
          break

      print("Increase complexity of the BERTopic")
      self.update_submodels(n_components = 2, min_cluster_size = cluster_size)
      cluster_size -= 50
      repeat = True

    self.berTopic_model.set_topic_labels(results)

    return results


  # BERTopic
  def bertTopic_fit(self):
    self.embeddings = self.embedding_model.encode(list(self.preprocessed_article_df.article), show_progress_bar=True)

    berTopic_model = BERTopic(
          #sub-models
          embedding_model = self.embedding_model,
          umap_model = self.umap_model,
          hdbscan_model = self.hdbscan_model,
          representation_model = self.representation_model,

          # Hyperparameters
          n_gram_range=(1, 3),
          verbose = True
      )

    topics, probs = berTopic_model.fit_transform(list(self.preprocessed_article_df.article), self.embeddings)

    info = berTopic_model.get_topic_info()
    averaged_keywords = self.concatKeywords(info)
    berTopic_model.set_topic_labels(averaged_keywords)
    self.berTopic_model = berTopic_model

    self.berTopic_df = berTopic_model.get_document_info(list(self.preprocessed_article_df.article))
    self.berTopic_df = self.berTopic_df.loc[:, ["Document", "CustomName"]]


  # Update UMAP and HDBSCAN Model
  def update_submodels(self, n_components, min_cluster_size):
      self.n_components = n_components
      self.min_cluster_size = min_cluster_size

      self.umap_model = UMAP(
          n_neighbors = self.n_neighbors,
          n_components = self.n_components,
          min_dist = 0.0,
          metric = "cosine",
          random_state=42
      )

      self.hdbscan_model = HDBSCAN(
          min_cluster_size = self.min_cluster_size,
          metric = "euclidean",
          cluster_selection_method = "eom",
          prediction_data = True
      )


  # Drop Topic name == "NA"
  def dropNATopics(self):
    indices = np.where(self.berTopic_df.CustomName == "NA")[0]
    self.article_df = self.article_df.drop(indices).reset_index(drop=True)
    self.preprocessed_article_df = self.preprocessed_article_df.drop(indices).reset_index(drop=True)
    self.berTopic_df = self.berTopic_df.drop(indices).reset_index(drop=True)



  # Filtering and Labeling
  def groqTopic_fit(self):
    # Topic Modeling
    G = GroqTopic(api_key=self.api_key, model= self.groq_model)
    # check if articles are written in English
    info = self.berTopic_model.get_topic_info()
    keywords_list = info.CustomName.to_list()
    keywords_list = pd.Series(keywords_list)

    G.fit_topic(keywords_list)
    topic_results = G.labeling_results

    self.berTopic_model.set_topic_labels(topic_results)
    self.berTopic_df = self.berTopic_model.get_document_info(list(self.preprocessed_article_df.article))
    self.berTopic_df = self.berTopic_df.loc[:, ["Document", "CustomName"]]

    return topic_results

In [ ]:
# id = preprocessed_df.id
# indices = np.where(df.id.isin(id))
# df = df.loc[indices].reset_index(drop=True)

In [ ]:
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, PartOfSpeech
from bertopic import BERTopic

In [ ]:
TNIE = TechNewsInsightEngine(df, api_key="",
                             groq_model="llama-3.2-90b-text-preview",
                             n_neighbors = 15,
                             n_components = 2,
                             min_cluster_size = 50,
                             top_n_keywords = 30,
                             embedding_model = "BAAI/bge-small-en")

TNIE.preprocessed_article_df = preprocessed_df

In [ ]:
results = TNIE.topic_fit()

Topic Modeling 2
Increase complexity of the BERTopic
